# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushdevo/10x.ai/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
One row represents a single URL's Google Search performance metrics for exactly one calendar day

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
fact_content_daily_performance

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
Time window: Mid-panel iteration month: 2026-03.

Predict or rank (label/proxy): Predicting if the URL will achieve a Top 5 average position in the subsequent 7 days (binary classification).

Deliberate exclusion: Excluding URLs with less than 10 total impressions in the trailing 7 days, as they represent unindexed or dead pages that skew the baseline.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# SETUP: Load the data using your Colab secret token
# ---------------------------------------------------------
hf_token = userdata.get('HF_TOKEN')

print("Loading warehouse data...")
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=False,
    token=hf_token
)
df = dataset.to_pandas()
df['date'] = pd.to_datetime(df['date'])

# Isolate the mid-panel month (March 2026) to protect the June test set
df_march = df[(df['date'] >= '2026-03-01') & (df['date'] <= '2026-03-31')].copy()

# ---------------------------------------------------------
# PART 2: Prove Three Facts
# ---------------------------------------------------------
print("\n--- 2. Proving Three Facts ---")

# Fact 1: Row count & Date span
row_count = len(df_march)
min_date, max_date = df_march['date'].min().date(), df_march['date'].max().date()
print(f"1. Date Span: {min_date} to {max_date} | Total Rows: {row_count}")

# Fact 2: Availability (Filter with IS TRUE)
# Note: If the column name differs in your specific schema, update 'is_indexed'
if 'is_indexed' in df_march.columns:
    df_march = df_march[df_march['is_indexed'] == True].copy()
    print(f"2. Availability: {len(df_march)} rows survive the 'IS TRUE' index filter.")

# Fact 3: The Grain (URL + Date is strictly unique)
grain_check = len(df_march.drop_duplicates(subset=['url', 'date'])) == len(df_march)
print(f"3. Grain Check (URL + Date is unique per row): {grain_check}")


# ---------------------------------------------------------
# PART 3: Five Features Frame
# ---------------------------------------------------------
print("\n--- 3. Five Features ---")
features = pd.DataFrame(index=df_march.index)

# 1. knowable at the decision moment because: Historical impressions up to yesterday are finalized in the warehouse.
features['f1_trailing_7d_impressions'] = df_march.get('impressions_7d_sum', 0)

# 2. knowable at the decision moment because: The daily BigQuery sync aggregates device logs prior to the prediction window.
features['f2_desktop_clicks'] = df_march.get('desktop_clicks', 0)

# 3. knowable at the decision moment because: Search Console position data for T-1 through T-3 is already calculated.
features['f3_avg_position_3d'] = df_march.get('position_3d_avg', 0)

# 4. knowable at the decision moment because: The URL's character length is a static property of the content.
features['f4_url_length'] = df_march['url'].str.len()

# 5. knowable at the decision moment because: It compares T-1 clicks to T-2 clicks, both of which are historical facts.
if 'clicks' in df_march.columns and 'clicks_prev_day' in df_march.columns:
    features['f5_daily_click_momentum'] = df_march['clicks'] - df_march['clicks_prev_day']
else:
    features['f5_daily_click_momentum'] = 0

print("Feature frame built successfully.")


# ---------------------------------------------------------
# PART 4: The Target Leakage Trap
# ---------------------------------------------------------
print("\n--- 4. The Leakage Trap ---")

# Honest Target: Will the URL average a Top 5 position over the next 7 days?
# (Using a dummy target if the exact next_7d column is missing for the test)
if 'position_next_7d_avg' in df_march.columns:
    y = (df_march['position_next_7d_avg'] <= 5.0).astype(int)
else:
    y = (df_march.get('clicks', 0) > 10).astype(int) # Fallback dummy target

# THE TRAP: We deliberately leak future data into the features
# We use next week's clicks, which we wouldn't know today.
features['LEAKED_clicks_future'] = df_march.get('clicks_next_7d', df_march.get('clicks', 0))

X = features.fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train WITH the leaked feature
model_leaked = RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42)
model_leaked.fit(X_train, y_train)
leaked_score = accuracy_score(y_test, model_leaked.predict(X_test))
print(f"Score WITH leaked future feature: {leaked_score:.4f} (The Trap)")

# Train WITHOUT the leaked feature
X_train_honest = X_train.drop(columns=['LEAKED_clicks_future'])
X_test_honest = X_test.drop(columns=['LEAKED_clicks_future'])

model_honest = RandomForestClassifier(n_estimators=10, max_depth=3, random_state=42)
model_honest.fit(X_train_honest, y_train)
honest_score = accuracy_score(y_test, model_honest.predict(X_test_honest))
print(f"Score WITHOUT leaked feature:   {honest_score:.4f} (The Honest Baseline)")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.